In [5]:
import os
import json
import requests
import subprocess

# ---- Заполни при необходимости ----
MLFLOW_TRACKING_URI = "http://192.168.49.1:5000"
MINIO_URL = "http://192.168.49.1:9091"
AIRFLOW_URL = "http://localhost:8080"
K8S_NAMESPACE = "seldon"
MODEL_NAME = "infra-network-degradation-demo"

# MinIO/S3 для MLflow artifact store
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["AWS_ACCESS_KEY_ID"] = "minio"       # поменяй если у тебя другие креды
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123"   # поменяй если у тебя другие креды
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://192.168.49.1:9090"  # S3 API endpoint MinIO

def check_http(name, url):
    try:
        r = requests.get(url, timeout=4)
        print(f"[OK] {name}: {url} -> {r.status_code}")
    except Exception as e:
        print(f"[ERR] {name}: {url} -> {e}")

check_http("MLflow", MLFLOW_TRACKING_URI)
check_http("MinIO Console", MINIO_URL)
check_http("Airflow", AIRFLOW_URL)

try:
    cluster = subprocess.check_output(["kubectl", "config", "current-context"], text=True).strip()
    print("[OK] kubectl context:", cluster)
except Exception as e:
    print("[ERR] kubectl not ready:", e)


[OK] MLflow: http://192.168.49.1:5000 -> 200
[OK] MinIO Console: http://192.168.49.1:9091 -> 200
[OK] Airflow: http://localhost:8080 -> 200
[OK] kubectl context: mlcluster


In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 3500

cpu = np.random.uniform(20, 100, n)
mem = np.random.uniform(30, 100, n)
latency = np.random.gamma(shape=2.5, scale=25, size=n)   # ms
packet_loss = np.random.beta(2, 18, n) * 10               # %
error_rate = np.random.beta(2, 12, n) * 5                 # %
connections = np.random.normal(700, 220, n).clip(50, 2000)

# Простая формула "риска": чем выше latency/loss/errors/load, тем выше шанс деградации.
score = (
    0.025 * latency +
    0.45 * packet_loss +
    0.65 * error_rate +
    0.015 * cpu +
    0.012 * mem +
    0.0008 * connections +
    np.random.normal(0, 0.8, n)
)

threshold = np.percentile(score, 68)
degraded = (score > threshold).astype(int)

df = pd.DataFrame({
    "cpu_usage": cpu,
    "memory_usage": mem,
    "latency_ms": latency,
    "packet_loss_pct": packet_loss,
    "error_rate_pct": error_rate,
    "active_connections": connections,
    "degraded": degraded,
})

print(df.head())
print("shape:", df.shape, "| positive_rate:", df["degraded"].mean().round(3))

   cpu_usage  memory_usage  latency_ms  packet_loss_pct  error_rate_pct  \
0  49.963210     50.056368   74.937277         2.584875        0.857393   
1  96.057145     86.184822   80.648619         0.335831        0.919705   
2  78.559515     99.806653   31.092560         0.166553        0.946619   
3  67.892679     32.101820   33.287459         0.854838        1.472730   
4  32.481491     92.815611   28.369969         0.409212        0.567559   

   active_connections  degraded  
0          829.092621         1  
1          636.588759         1  
2          585.608960         0  
3          992.881416         0  
4         1022.595854         0  
shape: (3500, 7) | positive_rate: 0.32


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

FEATURES = [
    "cpu_usage", "memory_usage", "latency_ms", "packet_loss_pct",
    "error_rate_pct", "active_connections"
]
TARGET = "degraded"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=200, random_state=42))
])

model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba)
print("ROC-AUC:", round(auc, 4))
print(classification_report(y_test, pred))

ROC-AUC: 0.8927
              precision    recall  f1-score   support

           0       0.84      0.91      0.88       476
           1       0.78      0.64      0.70       224

    accuracy                           0.83       700
   macro avg       0.81      0.78      0.79       700
weighted avg       0.82      0.83      0.82       700



In [6]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("infra_network_demo_experiment")

with mlflow.start_run(run_name="logreg_infra_demo") as run:
    run_id = run.info.run_id

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("features", ",".join(FEATURES))
    mlflow.log_param("test_size", 0.2)

    mlflow.log_metric("roc_auc", float(auc))
    mlflow.log_metric("accuracy", float(accuracy_score(y_test, pred)))
    mlflow.log_metric("f1", float(f1_score(y_test, pred)))

    signature = mlflow.models.infer_signature(X_train, model.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=X_train.head(3)
    )

    mlflow.set_tags({
        "project": "infra-ml-demo",
        "domain": "network-observability",
        "stage": "demo",
        "owner": "senior-ml-dev"
    })

print("run_id:", run_id)
print("logged_model_uri:", model_info.model_uri)

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
latest = client.get_latest_versions(MODEL_NAME)
model_version = max([int(v.version) for v in latest])
print("registered version:", model_version)

2026/04/29 16:19:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 16:19:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'infra-network-degradation-demo'.
2026/04/29 16:19:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: infra-network-degradation-demo, version 1


🏃 View run logreg_infra_demo at: http://192.168.49.1:5000/#/experiments/3/runs/8c97c622ffe94b2db21c526eff7c96f1
🧪 View experiment at: http://192.168.49.1:5000/#/experiments/3
run_id: 8c97c622ffe94b2db21c526eff7c96f1
logged_model_uri: models:/m-aba485cd9b5f491fb9c4a671bfcc0166
registered version: 1


Created version '1' of model 'infra-network-degradation-demo'.
/tmp/ipykernel_1042853/1868831835.py:40: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(MODEL_NAME)


In [7]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=model_version,
    stage="Production",
    archive_existing_versions=True,
)

client.set_model_version_tag(MODEL_NAME, str(model_version), "deployment", "seldon")
client.set_model_version_tag(MODEL_NAME, str(model_version), "validated", "true")

prod = client.get_model_version(MODEL_NAME, str(model_version))
print("Production model:", prod.name, "v" + prod.version, "stage=", prod.current_stage)

Production model: infra-network-degradation-demo v1 stage= Production


/tmp/ipykernel_1042853/3061227969.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [9]:
import os
import yaml
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
mv = client.get_model_version(MODEL_NAME, str(model_version))

# source содержит путь к артефактам версии модели (например s3://.../artifacts/model)
storage_uri = mv.source
print("storage_uri:", storage_uri)

sdep_name = "infra-degradation-predictor"

seldon_deployment = {
    "apiVersion": "machinelearning.seldon.io/v1",
    "kind": "SeldonDeployment",
    "metadata": {
        "name": sdep_name,
        "namespace": K8S_NAMESPACE
    },
    "spec": {
        "name": sdep_name,
        "predictors": [
            {
                "name": "default",
                "replicas": 1,
                "graph": {
                    "name": "model",
                    "implementation": "SKLEARN_SERVER",
                    "modelUri": storage_uri
                }
            }
        ]
    }
}

os.makedirs("artifacts", exist_ok=True)
seldon_yaml_path = "artifacts/seldon-deployment.yaml"
with open(seldon_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(seldon_deployment, f, sort_keys=False)

print("saved:", seldon_yaml_path)
print(open(seldon_yaml_path, encoding="utf-8").read())

storage_uri: models:/m-aba485cd9b5f491fb9c4a671bfcc0166
saved: artifacts/seldon-deployment.yaml
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: infra-degradation-predictor
  namespace: seldon
spec:
  name: infra-degradation-predictor
  predictors:
  - name: default
    replicas: 1
    graph:
      name: model
      implementation: SKLEARN_SERVER
      modelUri: models:/m-aba485cd9b5f491fb9c4a671bfcc0166



In [10]:
import subprocess
import time

subprocess.run(["kubectl", "create", "ns", K8S_NAMESPACE], check=False)
subprocess.run(["kubectl", "apply", "-f", "artifacts/seldon-deployment.yaml"], check=True)

print("Waiting for deployment to become ready...")
for i in range(24):
    out = subprocess.check_output(
        ["kubectl", "get", "seldondeployment", "infra-degradation-predictor", "-n", K8S_NAMESPACE, "-o", "jsonpath={.status.state}"],
        text=True
    ).strip()
    print(f"try={i+1}, state={out}")
    if out == "Available":
        break
    time.sleep(10)

subprocess.run(["kubectl", "get", "pods", "-n", K8S_NAMESPACE], check=False)
subprocess.run(["kubectl", "get", "seldondeployment", "-n", K8S_NAMESPACE], check=False)

Error from server (AlreadyExists): namespaces "seldon" already exists


seldondeployment.machinelearning.seldon.io/infra-degradation-predictor created
Waiting for deployment to become ready...
try=1, state=Creating
try=2, state=Creating
try=3, state=Creating
try=4, state=Creating
try=5, state=Creating
try=6, state=Creating
try=7, state=Creating
try=8, state=Creating
try=9, state=Creating
try=10, state=Creating
try=11, state=Creating
try=12, state=Creating
try=13, state=Creating
try=14, state=Creating
try=15, state=Creating
try=16, state=Creating


KeyboardInterrupt: 